<a href="https://colab.research.google.com/github/kijac/26_1_EEE_Project/blob/test/260406_yolov8n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# GPU 확인
!nvidia-smi

Mon Apr  6 09:02:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Ultralytics 설치
!pip install ultralytics==8.3.40 -q

from ultralytics import YOLO
import torch
import os
import yaml
import shutil
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 898.5/898.5 kB 23.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [ ]:
# ============================================
# 방법 1: zip 파일 직접 업로드
# ============================================
# Colab 파일 업로드
from google.colab import files
uploaded = files.upload()  # zip 파일 선택

# 업로드된 zip 파일명 자동 감지
zip_name = list(uploaded.keys())[0]
print(f"업로드된 파일: {zip_name}")

# 압축 해제
!mkdir -p /content/dataset
!unzip -q -o "{zip_name}" -d /content/dataset

# 폴더 구조 확인
!find /content/dataset -type d | head -20
print("\n--- 파일 수 ---")
!find /content/dataset -name "*.jpg" -o -name "*.png" | wc -l

Saving Safety Helmet.v5i.yolov8.zip to Safety Helmet.v5i.yolov8.zip
업로드된 파일: Safety Helmet.v5i.yolov8.zip
/content/dataset
/content/dataset/test
/content/dataset/test/labels
/content/dataset/test/images
/content/dataset/valid
/content/dataset/valid/labels
/content/dataset/valid/images
/content/dataset/train
/content/dataset/train/labels
/content/dataset/train/images

--- 파일 수 ---
5000


In [ ]:
# 기존 data.yaml 확인
yaml_path = "/content/dataset/data.yaml"
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("=== 원본 data.yaml ===")
print(yaml.dump(data_config, default_flow_style=False))

# head 클래스의 인덱스 찾기
original_names = data_config['names']
if isinstance(original_names, dict):
    head_idx = [k for k, v in original_names.items() if v == 'head']
    keep_classes = {k: v for k, v in original_names.items() if v != 'head'}
else:
    head_idx = [i for i, name in enumerate(original_names) if name == 'head']
    keep_classes = [name for name in original_names if name != 'head']

print(f"\nhead 클래스 인덱스: {head_idx}")
print(f"사용할 클래스: {keep_classes}")

=== 원본 data.yaml ===
names:
- head
- helmet
- person
- vest
nc: 4
roboflow:
  license: CC BY 4.0
  project: safety-helmet-4mhdt-m4od4
  url: https://universe.roboflow.com/s-workspace-31kqe/safety-helmet-4mhdt-m4od4/dataset/5
  version: 5
  workspace: s-workspace-31kqe
test: ../test/images
train: ../train/images
val: ../valid/images


head 클래스 인덱스: [0]
사용할 클래스: ['helmet', 'person', 'vest']


In [ ]:
# head 어노테이션 제거 + 클래스 인덱스 재매핑
import glob

# 원본 클래스 목록에서 매핑 테이블 생성
if isinstance(original_names, dict):
    old_names = original_names
else:
    old_names = {i: name for i, name in enumerate(original_names)}

# head가 아닌 클래스만 새 인덱스로 매핑
new_idx = 0
remap = {}  # old_idx -> new_idx
new_names = {}
for old_i in sorted(old_names.keys()):
    if old_names[old_i] != 'head':
        remap[old_i] = new_idx
        new_names[new_idx] = old_names[old_i]
        new_idx += 1

print(f"인덱스 매핑: {remap}")
print(f"새 클래스: {new_names}")

# 모든 라벨 파일 수정
label_dirs = glob.glob("/content/dataset/**/labels", recursive=True)
total_removed = 0
total_remapped = 0

for label_dir in label_dirs:
    for txt_file in glob.glob(os.path.join(label_dir, "*.txt")):
        with open(txt_file, 'r') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            old_class = int(parts[0])
            if old_class in remap:
                parts[0] = str(remap[old_class])
                new_lines.append(' '.join(parts) + '\n')
                total_remapped += 1
            else:
                total_removed += 1

        with open(txt_file, 'w') as f:
            f.writelines(new_lines)

print(f"\n제거된 head 어노테이션: {total_removed}개")
print(f"유지된 어노테이션: {total_remapped}개")

인덱스 매핑: {1: 0, 2: 1, 3: 2}
새 클래스: {0: 'helmet', 1: 'person', 2: 'vest'}

제거된 head 어노테이션: 4843개
유지된 어노테이션: 31861개


In [ ]:
# 수정된 data.yaml 저장
data_config['names'] = new_names
data_config['nc'] = len(new_names)

# 경로를 절대경로로 수정
data_config['path'] = '/content/dataset'
# train/val/test 경로가 상대경로인지 확인
for key in ['train', 'val', 'test']:
    if key in data_config:
        p = data_config[key]
        if not os.path.isabs(p):
            # 상대경로 유지 (path 기준)
            pass

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("=== 수정된 data.yaml ===")
with open(yaml_path, 'r') as f:
    print(f.read())

=== 수정된 data.yaml ===
names:
  0: helmet
  1: person
  2: vest
nc: 3
path: /content/dataset
roboflow:
  license: CC BY 4.0
  project: safety-helmet-4mhdt-m4od4
  url: https://universe.roboflow.com/s-workspace-31kqe/safety-helmet-4mhdt-m4od4/dataset/5
  version: 5
  workspace: s-workspace-31kqe
test: ../test/images
train: ../train/images
val: ../valid/images



In [ ]:
# 클래스별 어노테이션 분포 확인
from collections import Counter

for split in ['train', 'valid', 'test']:
    label_path = f"/content/dataset/{split}/labels"
    if not os.path.exists(label_path):
        # Roboflow 포맷에 따라 경로가 다를 수 있음
        label_path = f"/content/dataset/{split}/labels"

    if not os.path.exists(label_path):
        print(f"{split}: 폴더 없음")
        continue

    class_counts = Counter()
    n_images = len(glob.glob(os.path.join(label_path, "*.txt")))

    for txt_file in glob.glob(os.path.join(label_path, "*.txt")):
        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_counts[int(parts[0])] += 1

    print(f"\n[{split}] 이미지: {n_images}장")
    for cls_id in sorted(class_counts.keys()):
        cls_name = new_names.get(cls_id, f"unknown_{cls_id}")
        print(f"  {cls_name}: {class_counts[cls_id]}개")


[train] 이미지: 3500장
  helmet: 11012개
  person: 10016개
  vest: 1389개

[valid] 이미지: 1000장
  helmet: 3040개
  person: 2859개
  vest: 460개

[test] 이미지: 500장
  helmet: 1455개
  person: 1433개
  vest: 197개


In [ ]:
# YOLOv8n 모델 로드 (pretrained COCO weights)
model = YOLO('yolov8n.pt')

# =============================================
# 학습 파라미터
# =============================================
results = model.train(
    data=yaml_path,

    # --- 기본 설정 ---
    epochs=150,              # 데이터 적으니 충분히 돌림
    imgsz=640,               # 표준 입력 크기
    batch=16,                # T4 16GB 기준 안정적

    # --- Optimizer ---
    optimizer='AdamW',       # 소규모 데이터에서 SGD보다 안정적
    lr0=0.001,               # AdamW 초기 학습률
    lrf=0.01,                # 최종 학습률 비율 (lr0 * lrf)
    weight_decay=0.0005,     # 과적합 방지
    warmup_epochs=5,         # 초반 안정화
    warmup_momentum=0.8,

    # --- Augmentation (소규모 데이터 → 강하게) ---
    hsv_h=0.015,             # 색조 변환
    hsv_s=0.7,               # 채도 변환
    hsv_v=0.4,               # 명도 변환
    degrees=10.0,            # 회전 (건설현장: 카메라 약간 틀어질 수 있음)
    translate=0.2,           # 이동
    scale=0.5,               # 스케일 변환 (다양한 거리)
    shear=2.0,               # 전단
    flipud=0.1,              # 상하 반전 (약하게)
    fliplr=0.5,              # 좌우 반전
    mosaic=1.0,              # 모자이크 (소규모 데이터에 효과적)
    mixup=0.15,              # MixUp (과적합 억제)
    copy_paste=0.1,          # Copy-Paste augmentation
    erasing=0.3,             # Random Erasing (가림 상황 대응)

    # --- 학습 안정화 ---
    cos_lr=True,             # Cosine LR scheduler
    close_mosaic=15,         # 마지막 15 epoch: mosaic 끄고 fine-tune
    patience=30,             # Early stopping patience

    # --- Loss weights ---
    box=7.5,                 # Box loss 가중치
    cls=0.5,                 # Classification loss
    dfl=1.5,                 # Distribution focal loss

    # --- 기타 ---
    workers=2,               # Colab 환경 안정성
    project='/content/runs',
    name='ppe_yolov8n',
    exist_ok=True,
    pretrained=True,         # COCO pretrained (전이학습)
    verbose=True,
    seed=42,
)

100%|██████████| 6.25M/6.25M [00:00<00:00, 135MB/s]

New https://pypi.org/project/ultralytics/8.4.33 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)


engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/dataset/data.yaml, epochs=150, time=None, patience=30, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=2, project=/content/runs, name=ppe_yolov8n, exist_ok=True, pretrained=True, optimizer=AdamW, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=15, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_width=None, format=torchscript, keras=False, optimize=F

100%|██████████| 755k/755k [00:00<00:00, 34.7MB/s]


Overriding model.yaml nc=80 with nc=3

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     56/150      2.69G      1.585      1.254      1.482         97        640: 100%|██████████| 219/219 [01:02<00:00,  3.48it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 32/32 [00:08<00:00,  3.57it/s]

                   all       1000       6359      0.718      0.696      0.695      0.325



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     57/150      2.83G      1.587      1.252      1.476        108        640: 100%|██████████| 219/219 [01:02<00:00,  3.50it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):  28%|██▊       | 9/32 [00:02<00:05,  3.91it/s]

In [ ]:
# 학습 결과 시각화
from IPython.display import Image, display

result_dir = '/content/runs/ppe_yolov8n'

# 학습 곡선
if os.path.exists(f'{result_dir}/results.png'):
    display(Image(filename=f'{result_dir}/results.png', width=900))

# Confusion Matrix
if os.path.exists(f'{result_dir}/confusion_matrix_normalized.png'):
    print("\n=== Confusion Matrix (Normalized) ===")
    display(Image(filename=f'{result_dir}/confusion_matrix_normalized.png', width=600))

# PR Curve
if os.path.exists(f'{result_dir}/PR_curve.png'):
    print("\n=== PR Curve ===")
    display(Image(filename=f'{result_dir}/PR_curve.png', width=600))

In [ ]:
# 검증 이미지 확인
if os.path.exists(f'{result_dir}/val_batch0_pred.png'):
    print("=== Validation Predictions ===")
    display(Image(filename=f'{result_dir}/val_batch0_pred.png', width=800))

In [ ]:
# best 모델로 validation
best_model = YOLO(f'{result_dir}/weights/best.pt')
metrics = best_model.val(data=yaml_path, imgsz=640, batch=16)

print("\n" + "="*50)
print("최종 성능 요약")
print("="*50)
print(f"mAP@50     : {metrics.box.map50:.4f}")
print(f"mAP@50-95  : {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

print("\n--- 클래스별 AP@50 ---")
for i, ap in enumerate(metrics.box.ap50):
    print(f"  {new_names[i]:10s}: {ap:.4f}")

-------------------

In [ ]:
# Colab에서 웹캠 캡처 → 추론 → 결과 표시
# (Colab은 직접 cv2.VideoCapture가 안 되므로 JS로 캡처)

from IPython.display import display, Javascript, HTML, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import numpy as np
import cv2
import time

# 웹캠 캡처 JS 함수
def init_webcam():
    js = Javascript('''
        async function initCamera() {
            const div = document.createElement('div');
            const video = document.createElement('video');
            video.style.display = 'block';
            video.width = 640;
            video.height = 480;
            div.appendChild(video);

            const stream = await navigator.mediaDevices.getUserMedia(
                {video: {width: 640, height: 480}}
            );
            video.srcObject = stream;
            await video.play();

            // 캔버스로 프레임 캡처
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;

            // 글로벌에 저장
            window._video = video;
            window._canvas = canvas;

            return [video.videoWidth, video.videoHeight];
        }

        async function captureFrame() {
            const canvas = window._canvas;
            const video = window._video;
            canvas.getContext('2d').drawImage(video, 0, 0);
            return canvas.toDataURL('image/jpeg', 0.8);
        }

        async function stopCamera() {
            if (window._video && window._video.srcObject) {
                window._video.srcObject.getTracks().forEach(t => t.stop());
            }
        }
    ''')
    display(js)

init_webcam()
print("웹캠 JS 함수 로드 완료")

In [ ]:
# 카메라 시작
dims = eval_js('initCamera()')
print(f"카메라 해상도: {dims}")

In [ ]:
# 실시간 추론 루프
# 중지하려면 런타임 > 실행 중단 (Ctrl+M+I)

from IPython.display import clear_output
import PIL.Image
import io

# 클래스별 색상 (BGR)
CLASS_COLORS = {
    'person': (255, 128, 0),    # 주황
    'helmet': (0, 255, 0),      # 초록
    'vest':   (0, 200, 255),    # 노랑
}

best_model = YOLO(f'{result_dir}/weights/best.pt')

N_FRAMES = 100  # 테스트할 프레임 수 (무한 루프 원하면 while True로 변경)

try:
    for i in range(N_FRAMES):
        # JS에서 프레임 캡처
        data_url = eval_js('captureFrame()')

        # base64 → numpy array
        binary = b64decode(data_url.split(',')[1])
        img_array = np.frombuffer(binary, dtype=np.uint8)
        frame = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

        if frame is None:
            continue

        # 추론
        t0 = time.time()
        results = best_model.predict(
            frame,
            imgsz=640,
            conf=0.4,
            iou=0.5,
            verbose=False
        )
        dt = time.time() - t0
        fps = 1.0 / dt if dt > 0 else 0

        # 결과 그리기
        annotated = frame.copy()

        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = float(box.conf[0])
                cls_id = int(box.cls[0])
                cls_name = new_names.get(cls_id, f'cls_{cls_id}')
                color = CLASS_COLORS.get(cls_name, (255, 255, 255))

                cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
                label = f'{cls_name} {conf:.2f}'
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
                cv2.rectangle(annotated, (x1, y1-th-8), (x1+tw, y1), color, -1)
                cv2.putText(annotated, label, (x1, y1-5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 1)

        # FPS 표시
        cv2.putText(annotated, f'FPS: {fps:.1f}', (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # Colab에 표시 (BGR → RGB)
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        pil_img = PIL.Image.fromarray(annotated_rgb)

        # JPEG로 변환해서 표시
        buf = io.BytesIO()
        pil_img.save(buf, format='JPEG', quality=80)

        clear_output(wait=True)
        display(IPImage(data=buf.getvalue(), width=640))
        print(f'Frame {i+1}/{N_FRAMES} | FPS: {fps:.1f} | '
              f'Detections: {len(results[0].boxes) if results[0].boxes is not None else 0}')

except KeyboardInterrupt:
    print("\n중단됨")

finally:
    # 카메라 정리
    eval_js('stopCamera()')
    print("카메라 종료")

In [ ]:
# (선택) 단일 프레임 캡처 후 추론 결과 확인
# 실시간이 아니라 한 장만 찍어서 자세히 보고 싶을 때

data_url = eval_js('captureFrame()')
binary = b64decode(data_url.split(',')[1])
img_array = np.frombuffer(binary, dtype=np.uint8)
frame = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

results = best_model.predict(frame, imgsz=640, conf=0.3, verbose=False)

# Ultralytics 내장 시각화
annotated = results[0].plot()
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
display(PIL.Image.fromarray(annotated_rgb))

# 탐지 결과 상세
if results[0].boxes is not None:
    print(f"\n탐지된 객체: {len(results[0].boxes)}개")
    for box in results[0].boxes:
        cls_name = new_names.get(int(box.cls[0]), 'unknown')
        conf = float(box.conf[0])
        print(f"  {cls_name}: {conf:.3f}")

In [ ]:
# 카메라 종료
eval_js('stopCamera()')
print("카메라 종료 완료")

-----------------------------

In [ ]:
# ONNX export (Hailo DFC 변환을 위해)
best_model.export(
    format='onnx',
    imgsz=640,
    opset=13,           # Hailo 호환 opset
    simplify=True,      # ONNX simplifier 적용
    dynamic=False,      # 고정 입력 크기 (Edge 배포용)
)

print(f"\nONNX 파일: {result_dir}/weights/best.onnx")
onnx_size = os.path.getsize(f'{result_dir}/weights/best.onnx') / (1024*1024)
print(f"파일 크기: {onnx_size:.1f} MB")

In [ ]:
# Google Drive에 저장
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/PPE_Model'
os.makedirs(save_dir, exist_ok=True)

# best.pt, best.onnx, 학습 결과 복사
shutil.copy(f'{result_dir}/weights/best.pt', save_dir)
shutil.copy(f'{result_dir}/weights/best.onnx', save_dir)
shutil.copy(f'{result_dir}/results.csv', save_dir)
if os.path.exists(f'{result_dir}/results.png'):
    shutil.copy(f'{result_dir}/results.png', save_dir)

print(f"저장 완료: {save_dir}")
!ls -lh {save_dir}

In [ ]:
# 또는 직접 다운로드
# files.download(f'{result_dir}/weights/best.pt')
# files.download(f'{result_dir}/weights/best.onnx')